# Número Efetivo de Candidatos em Recursos (NECr)

Calcula o NECr por lista (partido × UF × ano) usando a fórmula Laakso-Taagepera (1979)
aplicada às proporções de recursos — equivalente ao índice de Herfindahl-Hirschman invertido:

$$\text{NECr} = \frac{1}{\sum_i p_i^2}$$

onde $p_i = $ `prop_vr_receita_candidato` (fatia do candidato $i$ no total FEFC/FP da lista).

O objetivo é produzir estatísticas descritivas para a **introdução do Capítulo 4**:
quantos candidatos recebem recursos de forma efetiva, e essa concentração está próxima do
patamar teórico de coordenação intrapartidária?

**Balizador correto: magnitude partidária ($M_p$), não magnitude do distrito.** A literatura
de coordenação intrapartidária em listas abertas (Crisp et al. 2007) prevê que o partido
concentra recursos em torno de $M_p + 1$ candidatos "viáveis", onde $M_p$ — a **magnitude
partidária** — é o número de cadeiras que o partido espera conquistar na UF, operacionalizado
ex-ante pela sua **bancada estadual prévia**: as cadeiras da legenda na UF no dia anterior ao
início do período de realização das convenções partidárias (`data/processed/bancada_partido_uf.csv`).

Isso é conceitualmente distinto da magnitude do distrito ($M$, `qt_vaga`) — o total de cadeiras
em disputa na UF, igual para todos os partidos da mesma UF e que não reflete a força eleitoral
específica de cada legenda. Uma versão anterior deste notebook usava incorretamente $M$
(`qt_vaga`) como balizador da regra $M+1$; esta versão recalcula tudo usando $M_p$.

In [1]:
import os
from pathlib import Path
ROOT = Path().resolve().parent  # notebooks/ -> project root
os.chdir(ROOT)


In [2]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path("src/2_gold").resolve()))
from cap3_cs_features import gerar_features

rrd_raw = pd.read_parquet("data/processed/rrd_df_novo.parquet")
rrd = rrd_raw[rrd_raw.ano_eleicao.isin([2018, 2022])].copy()
rrd = gerar_features(rrd)

print(f"Total de candidatos: {len(rrd):,}")
print(f"  2018: {(rrd.ano_eleicao == 2018).sum():,}")
print(f"  2022: {(rrd.ano_eleicao == 2022).sum():,}")

Total de candidatos: 17,305
  2018: 7,630
  2022: 9,675


## 1. Magnitude partidária ($M_p$): distribuição da bancada estadual prévia

$M_p$ é construída a partir de `bancada_partido_uf.csv` (gerado por
`src/0_bronze/bancada_por_partido_uf.py`, via API da Câmara dos Deputados, nas datas
2014-06-10 / 2018-07-20 / 2022-07-20 — véspera do início das convenções partidárias),
agregada ao nível de partido × UF × ano. Partidos sem registro no arquivo (sem deputados
eleitos pela legenda naquela UF na eleição anterior) recebem $M_p = 0$.

In [3]:
bancada = pd.read_csv("data/processed/bancada_partido_uf.csv")
harmonizacao = {"PP**": "PP", "PCdoB": "PC do B", "PTdoB": "PT do B", "SD": "SOLIDARIEDADE"}
bancada["sg_partido"] = bancada["sg_partido"].replace(harmonizacao)

bancada_uf = (
    bancada[bancada.ano_eleicao.isin([2018, 2022])]
    [["ano_eleicao", "sg_uf", "sg_partido", "n_deputados"]]
    .rename(columns={"n_deputados": "Mp"})
)

rrd = rrd.merge(bancada_uf, on=["ano_eleicao", "sg_uf", "sg_partido"], how="left")
rrd["Mp"] = rrd["Mp"].fillna(0).astype(int)

listas_mp = rrd.drop_duplicates(["ano_eleicao", "sg_uf", "sg_partido"])[
    ["ano_eleicao", "sg_uf", "sg_partido", "Mp"]
]

print(f"Listas (partido × UF × ano): {len(listas_mp):,}")
print(f"% de listas com Mp = 0 (partido sem bancada estadual prévia): {(listas_mp['Mp'] == 0).mean()*100:.1f}%\n")

print("Distribuição de Mp por ano:")
print(listas_mp.groupby("ano_eleicao")["Mp"].describe().round(2).to_string())

print("\nPercentis de Mp — todas as listas (incl. Mp=0):")
print(listas_mp["Mp"].quantile([.10, .25, .50, .75, .90, .95, .99]).round(1).to_string())

print("\nDistribuição de frequência de Mp (geral):")
print(listas_mp["Mp"].value_counts().sort_index().to_string())

Listas (partido × UF × ano): 1,570
% de listas com Mp = 0 (partido sem bancada estadual prévia): 65.4%

Distribuição de Mp por ano:
             count  mean   std  min  25%  50%  75%   max
ano_eleicao                                             
2018         859.0  0.59  1.23  0.0  0.0  0.0  1.0  13.0
2022         711.0  0.72  1.47  0.0  0.0  0.0  1.0  15.0

Percentis de Mp — todas as listas (incl. Mp=0):
0.10    0.0
0.25    0.0
0.50    0.0
0.75    1.0
0.90    2.0
0.95    3.0
0.99    7.0

Distribuição de frequência de Mp (geral):
Mp
0     1026
1      343
2       99
3       42
4       18
5       16
6        7
7       11
8        2
9        2
10       1
11       1
13       1
15       1


## 2. Cálculo do NECr por lista

In [4]:
def agregar_lista(g):
    total_rec = g["vr_receita_recursos_partidos"].sum()
    n_fundados = (g["vr_receita_recursos_partidos"] > 0).sum()
    sum_sq = (g["prop_vr_receita_candidato"] ** 2).sum()
    return pd.Series({
        "Mp":           g["Mp"].iloc[0],
        "qt_vaga":      g["qt_vaga"].iloc[0],
        "dm_cat":       g["dm_cat"].iloc[0],
        "n_cands":      len(g),
        "n_fundados":   int(n_fundados),
        "total_rec":    total_rec,
        "sum_sq_prop":  sum_sq,
        "NECr":         (1 / sum_sq) if sum_sq > 0 else np.nan,
    })

listas = (
    rrd
    .groupby(["ano_eleicao", "sg_uf", "sg_partido"], observed=True)
    .apply(agregar_lista, include_groups=False)
    .reset_index()
)

# Apenas listas com algum recurso distribuído
listas_rec = listas[listas["total_rec"] > 0].copy()

# Razões em relação à magnitude partidária (Mp). NECr/Mp é indefinida quando Mp=0
# (partido sem bancada estadual prévia); NECr/(Mp+1) é sempre bem definida e é a
# métrica focal, pois a regra M+1 (Cox 1997) prevê coordenação em torno de Mp+1
# candidatos viáveis mesmo quando o partido parte de bancada zero.
listas_rec["NECr_sobre_Mp"]   = np.where(listas_rec["Mp"] > 0, listas_rec["NECr"] / listas_rec["Mp"], np.nan)
listas_rec["NECr_sobre_Mpp1"] = listas_rec["NECr"] / (listas_rec["Mp"] + 1)
listas_rec["NECr_sobre_N"]    = listas_rec["NECr"] / listas_rec["n_cands"]

print(f"Listas com recursos: {len(listas_rec):,}")
print(f"  2018: {(listas_rec.ano_eleicao == 2018).sum():,}")
print(f"  2022: {(listas_rec.ano_eleicao == 2022).sum():,}")
print(f"\nListas sem nenhum recurso (excluídas): {listas['total_rec'].eq(0).sum()}")
print(f"Listas com recursos e Mp = 0: {(listas_rec['Mp'] == 0).sum():,} ({(listas_rec['Mp'] == 0).mean()*100:.1f}%)")

Listas com recursos: 1,434
  2018: 786
  2022: 648

Listas sem nenhum recurso (excluídas): 136
Listas com recursos e Mp = 0: 894 (62.3%)


## 3. Estatísticas descritivas gerais

In [5]:
def tabela_desc(series, label=""):
    s = series.dropna()
    return pd.Series({
        "n":      len(s),
        "média":  s.mean(),
        "mediana":s.median(),
        "dp":     s.std(),
        "p10":    s.quantile(0.10),
        "p25":    s.quantile(0.25),
        "p75":    s.quantile(0.75),
        "p90":    s.quantile(0.90),
        "min":    s.min(),
        "max":    s.max(),
    }, name=label)

desc_geral = pd.DataFrame([
    tabela_desc(listas_rec["NECr"],            "NECr"),
    tabela_desc(listas_rec["n_fundados"],      "N fundados"),
    tabela_desc(listas_rec["n_cands"],         "N candidatos"),
    tabela_desc(listas_rec["Mp"],              "Mp (bancada prévia)"),
    tabela_desc(listas_rec["NECr_sobre_Mp"],   "NECr/Mp (Mp>0)"),
    tabela_desc(listas_rec["NECr_sobre_Mpp1"], "NECr/(Mp+1)"),
    tabela_desc(listas_rec["NECr_sobre_N"],    "NECr/N"),
]).T

print("Estatísticas descritivas — todas as listas com recursos (2018+2022)\n")
print(desc_geral.round(2).to_string())

Estatísticas descritivas — todas as listas com recursos (2018+2022)

            NECr  N fundados  N candidatos  Mp (bancada prévia)  NECr/Mp (Mp>0)  NECr/(Mp+1)   NECr/N
n        1434.00     1434.00       1434.00              1434.00          540.00      1434.00  1434.00
média       4.29        9.88         11.67                 0.71            2.97         3.08     0.56
mediana     2.85        6.00          7.00                 0.00            2.10         1.90     0.52
dp          4.29       12.48         14.41                 1.39            2.54         3.46     0.30
p10         1.00        1.00          1.00                 0.00            1.00         0.77     0.17
p25         1.28        2.00          3.00                 0.00            1.21         1.00     0.32
p75         5.82       11.00         13.00                 1.00            3.70         3.72     0.82
p90         8.97       25.00         29.00                 2.00            6.35         7.08     1.00
min         1

## 4. Comparação 2018 × 2022

In [6]:
rows = []
for ano in [2018, 2022]:
    sub = listas_rec[listas_rec.ano_eleicao == ano]
    rows.append(tabela_desc(sub["NECr"],            f"NECr — {ano}"))
    rows.append(tabela_desc(sub["NECr_sobre_Mp"],    f"NECr/Mp — {ano}"))
    rows.append(tabela_desc(sub["NECr_sobre_Mpp1"],  f"NECr/(Mp+1) — {ano}"))
    rows.append(tabela_desc(sub["NECr_sobre_N"],     f"NECr/N — {ano}"))

by_ano = pd.DataFrame(rows).T
print("Estatísticas descritivas por ano\n")
print(by_ano.round(3).to_string())

Estatísticas descritivas por ano

         NECr — 2018  NECr/Mp — 2018  NECr/(Mp+1) — 2018  NECr/N — 2018  NECr — 2022  NECr/Mp — 2022  NECr/(Mp+1) — 2022  NECr/N — 2022
n            786.000         290.000             786.000        786.000      648.000         250.000             648.000        648.000
média          2.962           1.789               2.237          0.562        5.902           4.346               4.106          0.554
mediana        1.878           1.370               1.140          0.514        4.814           3.534               2.863          0.536
dp             3.170           1.258               2.785          0.314        4.875           2.927               3.899          0.272
p10            1.000           1.000               0.557          0.150        1.300           1.629               1.028          0.191
p25            1.000           1.002               1.000          0.292        2.471           2.294               1.742          0.333
p75           

## 5. NECr por magnitude do distrito (estratificação descritiva secundária)

`dm_cat` (pequeno/médio/grande, via `qt_vaga`) continua útil como controle descritivo do
tamanho do distrito, mas não é mais usado como balizador do NECr — apenas para checar se a
magnitude partidária mediana ($M_p$) varia com o tamanho do distrito.

In [7]:
ordem_dm = ["Pequeno (8–12)", "Médio (16–31)", "Grande (39–70)"]

rows_mag = []
for ano in [2018, 2022]:
    for dm in ordem_dm:
        sub = listas_rec[
            (listas_rec.ano_eleicao == ano) & (listas_rec.dm_cat == dm)
        ]
        if len(sub) == 0:
            continue
        label = f"{dm} | {ano}"
        rows_mag.append(pd.Series({
            "n_listas":   len(sub),
            "M_mediana":  sub["qt_vaga"].median(),
            "Mp_mediana": sub["Mp"].median(),
            "N_mediana":  sub["n_cands"].median(),
            "NECr_média": sub["NECr"].mean(),
            "NECr_mediana": sub["NECr"].median(),
            "NECr/(Mp+1)_média": sub["NECr_sobre_Mpp1"].mean(),
            "NECr/N_média": sub["NECr_sobre_N"].mean(),
        }, name=label))

tab_mag = pd.DataFrame(rows_mag).T
print("NECr por magnitude do distrito e ano\n")
print(tab_mag.round(3).to_string())

NECr por magnitude do distrito e ano

                   Pequeno (8–12) | 2018  Médio (16–31) | 2018  Grande (39–70) | 2018  Pequeno (8–12) | 2022  Médio (16–31) | 2022  Grande (39–70) | 2022
n_listas                         413.000               242.000                131.000                321.000               210.000                117.000
M_mediana                          8.000                22.000                 46.000                  8.000                22.000                 53.000
Mp_mediana                         0.000                 0.000                  1.000                  0.000                 0.000                  1.000
N_mediana                          3.000                 6.000                 22.000                  8.000                15.000                 39.000
NECr_média                         2.126                 3.091                  5.360                  4.224                 6.181                 10.006
NECr_mediana                       1.4

## 6. Distribuição NECr/($M_p$+1) — tabela de percentis por ano

Se coordenação = $M_p$+1, esperamos `NECr/(Mp+1)` ≈ 1. Como ~63% das listas têm $M_p$=0
(partido sem bancada estadual prévia), a tabela reporta a distribuição geral e também
separada entre listas com $M_p$=0 (sem bancada a defender) e $M_p$>0 (bancada a defender) —
é nesse segundo grupo que a hipótese de coordenação em torno de $M_p$+1 é diretamente
testável.

In [8]:
percentis = [0.10, 0.25, 0.50, 0.75, 0.90]

def descrever_mpp1(sub, titulo):
    s = sub["NECr_sobre_Mpp1"].dropna()
    print(f"\n{titulo} (n={len(s)}):")
    print(f"  média  = {s.mean():.3f}")
    print(f"  mediana= {s.median():.3f}")
    print(f"  dp     = {s.std():.3f}")
    qs = s.quantile(percentis)
    for p, v in zip(percentis, qs):
        print(f"  p{int(p*100):02d}    = {v:.3f}")
    below_1 = (s < 1).mean() * 100
    below_2 = (s < 2).mean() * 100
    print(f"  % listas com NECr < Mp+1     = {below_1:.1f}%")
    print(f"  % listas com NECr < 2(Mp+1)  = {below_2:.1f}%")

for ano in [2018, 2022]:
    sub = listas_rec[listas_rec.ano_eleicao == ano]
    print(f"\n{'='*60}\n{ano}\n{'='*60}")
    descrever_mpp1(sub, "Todas as listas")
    descrever_mpp1(sub[sub["Mp"] == 0], "Mp = 0 (sem bancada estadual prévia)")
    descrever_mpp1(sub[sub["Mp"] > 0],  "Mp > 0 (com bancada estadual prévia)")


2018

Todas as listas (n=786):
  média  = 2.237
  mediana= 1.140
  dp     = 2.785
  p10    = 0.557
  p25    = 1.000
  p50    = 1.140
  p75    = 2.280
  p90    = 4.971
  % listas com NECr < Mp+1     = 23.7%
  % listas com NECr < 2(Mp+1)  = 70.2%

Mp = 0 (sem bancada estadual prévia) (n=496):
  média  = 2.951
  mediana= 1.869
  dp     = 3.262
  p10    = 1.000
  p25    = 1.000
  p50    = 1.869
  p75    = 3.469
  p90    = 6.399
  % listas com NECr < Mp+1     = 0.0%
  % listas com NECr < 2(Mp+1)  = 56.7%

Mp > 0 (com bancada estadual prévia) (n=290):
  média  = 1.017
  mediana= 0.823
  dp     = 0.688
  p10    = 0.500
  p25    = 0.526
  p50    = 0.823
  p75    = 1.232
  p90    = 1.806
  % listas com NECr < Mp+1     = 64.1%
  % listas com NECr < 2(Mp+1)  = 93.4%

2022

Todas as listas (n=648):
  média  = 4.106
  mediana= 2.863
  dp     = 3.899
  p10    = 1.028
  p25    = 1.742
  p50    = 2.863
  p75    = 5.370
  p90    = 8.193
  % listas com NECr < Mp+1     = 4.0%
  % listas com NECr < 2(Mp+

## 7. Tabela resumo para o texto — NECr médio e mediano por magnitude do distrito × ano

In [9]:
tab_resumo = (
    listas_rec
    .groupby(["ano_eleicao", "dm_cat"], observed=True)
    .agg(
        n_listas=("NECr", "count"),
        Mp_med=("Mp", "median"),
        N_med=("n_cands", "median"),
        nfund_med=("n_fundados", "median"),
        NECr_media=("NECr", "mean"),
        NECr_med=("NECr", "median"),
        Mpp1_media=("NECr_sobre_Mpp1", "mean"),
        Mpp1_med=("NECr_sobre_Mpp1", "median"),
    )
    .reset_index()
)

# Linha de total
tot = (
    listas_rec
    .groupby("ano_eleicao")
    .agg(
        n_listas=("NECr", "count"),
        Mp_med=("Mp", "median"),
        N_med=("n_cands", "median"),
        nfund_med=("n_fundados", "median"),
        NECr_media=("NECr", "mean"),
        NECr_med=("NECr", "median"),
        Mpp1_media=("NECr_sobre_Mpp1", "mean"),
        Mpp1_med=("NECr_sobre_Mpp1", "median"),
    )
    .reset_index()
    .assign(dm_cat="Total")
)

tab_final = pd.concat([tab_resumo, tot], ignore_index=True)
tab_final["dm_cat"] = pd.Categorical(
    tab_final["dm_cat"],
    categories=["Pequeno (8–12)", "Médio (16–31)", "Grande (39–70)", "Total"],
    ordered=True,
)
tab_final = tab_final.sort_values(["ano_eleicao", "dm_cat"]).reset_index(drop=True)

print("Tabela-resumo: NECr por magnitude do distrito × ano")
print("Colunas: n=listas, Mp=magnitude partidária mediana, N=candidatos mediana,")
print("         Nfund=fundados mediana, NECr(m)=mediana, NECr/(Mp+1)(m)=mediana\n")
print(
    tab_final
    .rename(columns={
        "ano_eleicao": "Ano", "dm_cat": "Magnitude do distrito",
        "n_listas": "n", "Mp_med": "Mp", "N_med": "N", "nfund_med": "Nfund",
        "NECr_media": "NECr(μ)", "NECr_med": "NECr(m)",
        "Mpp1_media": "NECr/(Mp+1)(μ)", "Mpp1_med": "NECr/(Mp+1)(m)",
    })
    .round(2)
    .to_string(index=False)
)

Tabela-resumo: NECr por magnitude do distrito × ano
Colunas: n=listas, Mp=magnitude partidária mediana, N=candidatos mediana,
         Nfund=fundados mediana, NECr(m)=mediana, NECr/(Mp+1)(m)=mediana

 Ano Magnitude do distrito   n  Mp    N  Nfund  NECr(μ)  NECr(m)  NECr/(Mp+1)(μ)  NECr/(Mp+1)(m)
2018        Pequeno (8–12) 413 0.0  3.0    2.0     2.13     1.45            1.88            1.06
2018         Médio (16–31) 242 0.0  6.0    4.0     3.09     1.92            2.38            1.10
2018        Grande (39–70) 131 1.0 22.0   14.0     5.36     4.08            3.10            1.76
2018                 Total 786 0.0  4.0    3.0     2.96     1.88            2.24            1.14
2022        Pequeno (8–12) 321 0.0  8.0    7.0     4.22     3.84            3.43            2.80
2022         Médio (16–31) 210 0.0 15.0   14.0     6.18     5.68            4.42            2.79
2022        Grande (39–70) 117 1.0 39.0   35.0    10.01     8.27            5.40            3.09
2022                 Tot

## 8. Proporção de listas abaixo de $M_p$+1 e 2×($M_p$+1) — por magnitude do distrito × ano

In [10]:
resultados = []
for ano in [2018, 2022]:
    for dm in ordem_dm + ["Total"]:
        if dm == "Total":
            sub = listas_rec[listas_rec.ano_eleicao == ano]
        else:
            sub = listas_rec[
                (listas_rec.ano_eleicao == ano) & (listas_rec.dm_cat == dm)
            ]
        if len(sub) == 0:
            continue
        mpp1 = sub["Mp"] + 1
        pct_below_mpp1 = ((sub["NECr"] < mpp1).sum() / len(sub)) * 100
        pct_below_2mpp1 = ((sub["NECr"] < 2 * mpp1).sum() / len(sub)) * 100
        resultados.append({
            "Ano": ano, "Magnitude": dm,
            "n": len(sub),
            "% NECr < Mp+1":   round(pct_below_mpp1, 1),
            "% NECr < 2(Mp+1)": round(pct_below_2mpp1, 1),
        })

prop_tab = pd.DataFrame(resultados)
print("Proporção de listas com NECr abaixo dos limiares teóricos\n")
print(prop_tab.to_string(index=False))

Proporção de listas com NECr abaixo dos limiares teóricos

 Ano      Magnitude   n  % NECr < Mp+1  % NECr < 2(Mp+1)
2018 Pequeno (8–12) 413           21.3              72.6
2018  Médio (16–31) 242           30.6              73.6
2018 Grande (39–70) 131           18.3              56.5
2018          Total 786           23.7              70.2
2022 Pequeno (8–12) 321            2.8              35.5
2022  Médio (16–31) 210            5.7              36.7
2022 Grande (39–70) 117            4.3              29.1
2022          Total 648            4.0              34.7


## 9. Exemplos ilustrativos: listas típicas por magnitude do distrito

Mostra algumas listas com `NECr/(Mp+1)` próximo da mediana para uso descritivo no texto.

In [11]:
for ano in [2018, 2022]:
    print(f"\n{'='*70}")
    print(f"  {ano} — listas próximas da mediana de NECr/(Mp+1) por magnitude do distrito")
    print(f"{'='*70}")
    for dm in ordem_dm:
        sub = listas_rec[
            (listas_rec.ano_eleicao == ano) & (listas_rec.dm_cat == dm)
        ].copy()
        if len(sub) == 0:
            continue
        med = sub["NECr_sobre_Mpp1"].median()
        # 3 listas mais próximas da mediana
        sub["dist"] = (sub["NECr_sobre_Mpp1"] - med).abs()
        exs = sub.nsmallest(3, "dist")[["sg_uf", "sg_partido", "Mp", "qt_vaga", "n_cands", "n_fundados", "NECr", "NECr_sobre_Mpp1"]]
        print(f"\n  {dm} (mediana NECr/(Mp+1) = {med:.2f}):")
        print(exs.round(2).to_string(index=False))


  2018 — listas próximas da mediana de NECr/(Mp+1) por magnitude do distrito

  Pequeno (8–12) (mediana NECr/(Mp+1) = 1.06):
sg_uf sg_partido  Mp  qt_vaga  n_cands  n_fundados  NECr  NECr_sobre_Mpp1
   SE    PC do B   0        8        2           2  1.06             1.06
   RR        PMB   0        8        2           2  1.05             1.05
   PI     AVANTE   0       10        2           2  1.06             1.06

  Médio (16–31) (mediana NECr/(Mp+1) = 1.10):
sg_uf sg_partido  Mp  qt_vaga  n_cands  n_fundados  NECr  NECr_sobre_Mpp1
   MA        PMB   0       18        3           2  1.10             1.10
   RS       PSDB   1       31        7           7  2.21             1.11
   GO        PRP   0       17        4           4  1.11             1.11

  Grande (39–70) (mediana NECr/(Mp+1) = 1.76):
sg_uf sg_partido  Mp  qt_vaga  n_cands  n_fundados  NECr  NECr_sobre_Mpp1
   SP        PCB   0       70        2           2  1.76             1.76
   BA        MDB   1       39       36 

## 10. NECr × N fundados × $M_p$ — correlação e dispersão

In [12]:
for ano in [2018, 2022]:
    sub = listas_rec[listas_rec.ano_eleicao == ano]
    corr_n = sub[["NECr", "n_fundados", "Mp", "qt_vaga", "n_cands"]].corr()
    print(f"\n{ano} — correlações com NECr:")
    print(f"  NECr × N fundados      : {corr_n.loc['NECr','n_fundados']:.3f}")
    print(f"  NECr × Mp (bancada)    : {corr_n.loc['NECr','Mp']:.3f}")
    print(f"  NECr × M (qt_vaga)     : {corr_n.loc['NECr','qt_vaga']:.3f}")
    print(f"  NECr × N cands         : {corr_n.loc['NECr','n_cands']:.3f}")


2018 — correlações com NECr:
  NECr × N fundados      : 0.630
  NECr × Mp (bancada)    : 0.345
  NECr × M (qt_vaga)     : 0.437
  NECr × N cands         : 0.568

2022 — correlações com NECr:
  NECr × N fundados      : 0.721
  NECr × Mp (bancada)    : 0.433
  NECr × M (qt_vaga)     : 0.466
  NECr × N cands         : 0.674


## 11. NECr por magnitude do distrito × tipo de partido (ex-ante) × ano

`tipo_partido` é um conceito diferente de $M_p$: classifica o partido como "Competitivo" se
sua **bancada nacional** prévia (soma de `n_deputados` em todas as UFs) é ≥ 20 cadeiras —
critério de força nacional do partido, não a magnitude partidária estadual usada nas seções
anteriores. Usa `bancada_partido_uf.csv` (já carregado na Seção 1) — igual ao critério de
`2026-05-24_tabela_competitivos.ipynb`.

In [13]:
# --- tipo de partido ex-ante via bancada nacional (bancada já carregada/harmonizada na Seção 1) ---
bancada_nac = (
    bancada
    .groupby(["ano_eleicao", "sg_partido"])["n_deputados"]
    .sum()
    .reset_index()
    .rename(columns={"n_deputados": "bancada_nac"})
)

listas_tp = listas_rec.merge(bancada_nac, on=["ano_eleicao", "sg_partido"], how="left")
listas_tp["bancada_nac"] = listas_tp["bancada_nac"].fillna(0)
listas_tp["tipo_partido"] = np.where(
    listas_tp["bancada_nac"] >= 20, "Competitivo", "Menos competitivo"
)

# Verificação: PSL 2018 deve ser Menos competitivo; PL 2022 deve ser Competitivo
chk = (
    listas_tp[listas_tp.sg_partido.isin(["PSL", "PL"])]
    .drop_duplicates(["ano_eleicao", "sg_partido"])
    [["ano_eleicao", "sg_partido", "bancada_nac", "tipo_partido"]]
)
print(chk.to_string(index=False))

 ano_eleicao sg_partido  bancada_nac      tipo_partido
        2018        PSL          8.0 Menos competitivo
        2022         PL         77.0       Competitivo


In [14]:
def resumo_grupo(sub):
    return pd.Series({
        "n":       len(sub),
        "M":       sub["qt_vaga"].median(),
        "N":       sub["n_cands"].median(),
        "Nfund":   sub["n_fundados"].median(),
        "NECr(μ)": sub["NECr"].mean(),
        "NECr(m)": sub["NECr"].median(),
    })

ordem_dm = ["Pequeno (8–12)", "Médio (16–31)", "Grande (39–70)"]
ordem_tp = ["Competitivo", "Menos competitivo"]

rows = []
for ano in [2018, 2022]:
    sub_ano = listas_tp[listas_tp.ano_eleicao == ano]
    for dm in ordem_dm:
        sub_dm = sub_ano[sub_ano.dm_cat == dm]
        if len(sub_dm) == 0:
            continue
        for tp in ordem_tp:
            sub = sub_dm[sub_dm.tipo_partido == tp]
            if len(sub) == 0:
                continue
            rows.append({"Ano": ano, "Magnitude": dm, "Partido": tp, **resumo_grupo(sub).to_dict()})
        rows.append({"Ano": ano, "Magnitude": dm, "Partido": "Subtotal", **resumo_grupo(sub_dm).to_dict()})
    rows.append({"Ano": ano, "Magnitude": "Total", "Partido": "", **resumo_grupo(sub_ano).to_dict()})

tab_tp = pd.DataFrame(rows)
tab_tp["n"]       = tab_tp["n"].astype(int)
tab_tp["M"]       = tab_tp["M"].round(0).astype(int)
tab_tp["N"]       = tab_tp["N"].round(0).astype(int)
tab_tp["Nfund"]   = tab_tp["Nfund"].round(1)
tab_tp["NECr(μ)"] = tab_tp["NECr(μ)"].round(2)
tab_tp["NECr(m)"] = tab_tp["NECr(m)"].round(2)

print("NECr por magnitude × tipo de partido × ano\n")
print(tab_tp.to_string(index=False))

NECr por magnitude × tipo de partido × ano

 Ano      Magnitude           Partido   n  M  N  Nfund  NECr(μ)  NECr(m)
2018 Pequeno (8–12)       Competitivo 128  8  3    2.0     1.78     1.48
2018 Pequeno (8–12) Menos competitivo 285  8  3    2.0     2.28     1.45
2018 Pequeno (8–12)          Subtotal 413  8  3    2.0     2.13     1.45
2018  Médio (16–31)       Competitivo  72 20  6    5.0     2.99     2.37
2018  Médio (16–31) Menos competitivo 170 22  5    3.0     3.13     1.63
2018  Médio (16–31)          Subtotal 242 22  6    4.0     3.09     1.92
2018 Grande (39–70)       Competitivo  36 50 20   16.5     7.76     7.51
2018 Grande (39–70) Menos competitivo  95 46 25   14.0     4.45     2.88
2018 Grande (39–70)          Subtotal 131 46 22   14.0     5.36     4.08
2018          Total                   786 12  4    3.0     2.96     1.88
2022 Pequeno (8–12)       Competitivo 120  8  9    9.0     4.90     5.22
2022 Pequeno (8–12) Menos competitivo 201  8  6    6.0     3.82     3.25
2022 Pe

In [15]:
from tabulate import tabulate

tab_md = tab_tp.rename(columns={
    "Partido":  "Tipo de partido",
    "Nfund":    "N~fund~",
    "NECr(μ)":  "NECr (média)",
    "NECr(m)":  "NECr (mediana)",
}).copy()

# Pré-formatar floats para controlar casas decimais por coluna
tab_md["N~fund~"]        = tab_md["N~fund~"].apply(lambda x: f"{x:.1f}")
tab_md["NECr (média)"]   = tab_md["NECr (média)"].apply(lambda x: f"{x:.2f}")
tab_md["NECr (mediana)"] = tab_md["NECr (mediana)"].apply(lambda x: f"{x:.2f}")

print(tabulate(tab_md, headers="keys", tablefmt="pipe", showindex=False))

|   Ano | Magnitude      | Tipo de partido   |   n |   M |   N |   N~fund~ |   NECr (média) |   NECr (mediana) |
|------:|:---------------|:------------------|----:|----:|----:|----------:|---------------:|-----------------:|
|  2018 | Pequeno (8–12) | Competitivo       | 128 |   8 |   3 |       2   |           1.78 |             1.48 |
|  2018 | Pequeno (8–12) | Menos competitivo | 285 |   8 |   3 |       2   |           2.28 |             1.45 |
|  2018 | Pequeno (8–12) | Subtotal          | 413 |   8 |   3 |       2   |           2.13 |             1.45 |
|  2018 | Médio (16–31)  | Competitivo       |  72 |  20 |   6 |       5   |           2.99 |             2.37 |
|  2018 | Médio (16–31)  | Menos competitivo | 170 |  22 |   5 |       3   |           3.13 |             1.63 |
|  2018 | Médio (16–31)  | Subtotal          | 242 |  22 |   6 |       4   |           3.09 |             1.92 |
|  2018 | Grande (39–70) | Competitivo       |  36 |  50 |  20 |      16.5 |           7.76 |   

## 12. Notas de interpretação

**Interpretação central:**
- `NECr` é o equivalente ao N efetivo de partidos de Laakso-Taagepera, mas aplicado à distribuição de recursos dentro de uma lista.
- `NECr = 1` → um único candidato recebe 100% dos recursos.
- `NECr = N` → recursos perfeitamente igualados entre todos os candidatos.
- **Hipótese do $M_p$+1 (Cox 1997; Crisp et al. 2007):** se o partido coordena para defender/expandir sua bancada, concentra recursos em torno de $M_p$+1 candidatos viáveis, onde $M_p$ é a magnitude partidária (bancada estadual prévia). Portanto, esperamos `NECr/(Mp+1) ≈ 1`.
- Valores de `NECr/(Mp+1) < 1` indicam concentração mais intensa que o previsto (hiperseleção).
- Valores de `NECr/(Mp+1) > 1` indicam diluição de recursos além do número de candidatos viáveis esperado pela bancada prévia.

**Mp = 0 é o caso mais comum (~63% das listas), e é onde a regra falha mais:** partidos sem
bancada estadual prévia (Seção 6) têm `NECr/(Mp+1)` bem acima de 1 (mediana 1,87 em 2018 e
3,91 em 2022) — ou seja, pulverizam recursos entre vários candidatos "azarões" em vez de
concentrar em um único puxador, mesmo quando a previsão teórica para $M_p$=0 é justamente
$M_p$+1=1. Já entre listas com $M_p$>0 (partido já tem bancada a defender), a razão fica bem
mais próxima — e em 2018 até abaixo — de 1 (mediana 0,82 em 2018; 2,14 em 2022): é nesse
subgrupo que a regra $M_p$+1 tem poder explicativo real, e a atenuação 2018→2022 (0,82→2,14)
acompanha o mesmo padrão de atenuação já documentado no restante da tese para outras medidas
de coordenação. Isso é consistente com a teoria: partidos sem mandatos prévios na UF têm menos
informação para distinguir candidatos "viáveis" de "azarões" e, portanto, menos capacidade (ou
incentivo) de concentrar recursos ex-ante.

**Atenção:**
- Candidatos com `vr_receita_recursos_partidos = 0` têm `prop = 0`, logo `p² = 0`, e não contribuem para o denominador do NECr.
- O NECr mede a concentração **entre quem recebeu recursos**, não entre todos os candidatos da lista.
- Compare com `n_fundados` para ver quantos candidatos efetivamente têm `p > 0`.
- `dm_cat` (magnitude do distrito, via `qt_vaga`) é mantida apenas como estratificador descritivo secundário — não é mais usada como balizador de nenhuma razão NECr/M.

## 13. Decomposição do NECr sob abundância de recursos: piso distribuído vs. excedente estratégico

O FEFC cresceu de ~R$1,72 bi (2018) para ~R$4,96 bi (2022), aumento nominal de ~144%. Como o
NECr é invariante a escala (usa apenas proporções internas da lista), esse crescimento não
altera o NECr mecanicamente — mas pode alterar a **margem extensiva**: mais candidatos passam
a receber algum recurso, ampliando o denominador substantivo da distribuição sem que os
recursos eleitoralmente decisivos tenham deixado de ser concentrados nos candidatos
prioritários. As métricas abaixo separam essas duas margens:

- **`share_top_mpp1`**: participação dos $M_p$+1 candidatos mais financiados no total da lista.
- **`NECr_0/005/01/02/05`**: NECr recalculado ignorando candidatos com menos de 0%, 0,5%, 1%,
  2% e 5% do total da lista (proporções originais, não renormalizadas). Se o NECr bruto cai
  de volta para perto de $M_p$+1 conforme o corte aumenta, a elevação do NECr bruto reflete
  sobretudo uma cauda de candidaturas com recursos marginais.
- **`piso` / `necr_excedente`**: decompõe os recursos de cada candidato em um piso
  distributivo da lista (p10 dos valores positivos) mais um excedente; `necr_excedente` é o
  NECr calculado apenas sobre o excedente — a parte dos recursos que de fato diferencia
  candidatos entre si.
- **`necr_fefc` / `necr_fp`**: NECr calculado separadamente para recursos do FEFC e do Fundo
  Partidário, já que o argumento da abundância é especificamente sobre o desenho do FEFC.


In [16]:
from cap3_necr_decomposicao import decompor_lista, CORTES_RELEVANCIA

decomp = (
    rrd
    .groupby(["ano_eleicao", "sg_uf", "sg_partido"], observed=True)
    .apply(decompor_lista, include_groups=False)
    .reset_index()
)

listas_dec = listas_rec.merge(decomp, on=["ano_eleicao", "sg_uf", "sg_partido"], how="left")
listas_dec["necr_excedente_sobre_Mpp1"] = listas_dec["necr_excedente"] / (listas_dec["Mp"] + 1)

print(f"Listas com decomposição calculada: {len(listas_dec):,}")

Listas com decomposição calculada: 1,434


### 13.1 Participação dos $M_p$+1 mais financiados no total da lista

In [17]:
tab_top = pd.DataFrame([
    tabela_desc(listas_dec.loc[listas_dec.ano_eleicao == ano, "share_top_mpp1"], f"share_top_(Mp+1) - {ano}")
    for ano in [2018, 2022]
]).T

print("Participação dos Mp+1 candidatos mais financiados no total de recursos da lista")
print()
print(tab_top.round(3).to_string())

Participação dos Mp+1 candidatos mais financiados no total de recursos da lista

         share_top_(Mp+1) - 2018  share_top_(Mp+1) - 2022
n                        786.000                  648.000
média                      0.751                    0.514
mediana                    0.880                    0.480
dp                         0.283                    0.260
p10                        0.293                    0.192
p25                        0.539                    0.301
p75                        1.000                    0.703
p90                        1.000                    0.947
min                        0.035                    0.058
max                        1.000                    1.000


### 13.2 NECr com cortes de relevância mínima, vs. $M_p$+1

In [18]:
rows = []
for ano in [2018, 2022]:
    sub = listas_dec[listas_dec.ano_eleicao == ano]
    for label in CORTES_RELEVANCIA:
        col = f"NECr_{label}"
        razao = sub[col] / (sub["Mp"] + 1)
        rows.append(pd.Series({
            "ano": ano,
            "corte": col,
            "NECr_médio": sub[col].mean(),
            "NECr_mediano": sub[col].median(),
            "razão_Mp+1_média": razao.mean(),
            "razão_Mp+1_mediana": razao.median(),
        }))

tab_cortes = pd.DataFrame(rows)
print("NECr recalculado com limiares de relevância mínima (proporções originais)")
print()
print(tab_cortes.round(3).to_string(index=False))

NECr recalculado com limiares de relevância mínima (proporções originais)

 ano    corte  NECr_médio  NECr_mediano  razão_Mp+1_média  razão_Mp+1_mediana
2018   NECr_0       2.962         1.878             2.237               1.140
2018 NECr_005       2.963         1.878             2.237               1.140
2018  NECr_01       2.966         1.878             2.240               1.140
2018  NECr_02       2.979         1.878             2.249               1.142
2018  NECr_05       3.078         1.878             2.313               1.144
2022   NECr_0       5.902         4.814             4.106               2.863
2022 NECr_005       5.905         4.814             4.107               2.864
2022  NECr_01       5.919         4.814             4.114               2.866
2022  NECr_02       5.989         4.821             4.155               2.885
2022  NECr_05       7.528         4.857             5.084               2.937


### 13.3 Decomposição piso/excedente: NECr do excedente estratégico, vs. $M_p$+1

In [19]:
rows = []
for ano in [2018, 2022]:
    sub = listas_dec[listas_dec.ano_eleicao == ano]
    rows.append(pd.Series({
        "ano": ano,
        "piso_mediano_R$": sub["piso"].median(),
        "NECr_bruto_mediano": sub["NECr"].median(),
        "NECr_excedente_mediano": sub["necr_excedente"].median(),
        "NECr_bruto/(Mp+1)_mediano": sub["NECr_sobre_Mpp1"].median(),
        "NECr_excedente/(Mp+1)_mediano": sub["necr_excedente_sobre_Mpp1"].median(),
    }))

tab_exc = pd.DataFrame(rows)
print("NECr bruto vs. NECr do excedente acima do piso distributivo da lista (piso = p10 dos valores positivos)")
print()
print(tab_exc.round(3).to_string(index=False))

NECr bruto vs. NECr do excedente acima do piso distributivo da lista (piso = p10 dos valores positivos)

   ano  piso_mediano_R$  NECr_bruto_mediano  NECr_excedente_mediano  NECr_bruto/(Mp+1)_mediano  NECr_excedente/(Mp+1)_mediano
2018.0         20050.00               1.878                   1.886                      1.140                          1.119
2022.0         36722.22               4.814                   3.745                      2.863                          2.107


### 13.4 NECr por fonte: FEFC vs. Fundo Partidário

In [20]:
rows = []
for ano in [2018, 2022]:
    sub = listas_dec[listas_dec.ano_eleicao == ano]
    rows.append(pd.Series({
        "ano": ano,
        "NECr_FEFC_mediano": sub["necr_fefc"].median(),
        "NECr_FP_mediano": sub["necr_fp"].median(),
        "NECr_total_mediano": sub["NECr"].median(),
    }))

tab_fonte = pd.DataFrame(rows)
print("NECr mediano por fonte de recurso partidário")
print()
print(tab_fonte.round(3).to_string(index=False))

NECr mediano por fonte de recurso partidário

   ano  NECr_FEFC_mediano  NECr_FP_mediano  NECr_total_mediano
2018.0              1.671            1.293               1.878
2022.0              4.660            1.971               4.814


### 13.5 Notas de interpretação

- Se `NECr_0` (bruto) subir de 2018 para 2022 mas `NECr_05` (só candidatos com ≥5% da lista)
  permanecer estável e próximo de $M_p$+1, a elevação do NECr bruto é explicada por uma cauda
  de candidaturas com recursos marginais — não por diluição dos recursos estrategicamente
  relevantes.
- Da mesma forma, se `NECr_excedente/(Mp+1)` permanecer mais estável entre 2018 e 2022 do que
  `NECr_bruto/(Mp+1)`, isso indica que a abundância do FEFC ampliou principalmente o piso
  distribuído a candidaturas periféricas, preservando a concentração do excedente estratégico
  em torno do núcleo de candidatos prioritários.
- A comparação `necr_fefc` vs. `necr_fp` isola se o padrão de diluição é específico do desenho
  do FEFC (crescimento nominal muito maior que o Fundo Partidário) ou se é um padrão geral de
  todos os recursos partidários.
- Essas métricas são complementares ao NECr bruto (Seções 2–12), não substitutas: o NECr bruto
  continua sendo a medida canônica de concentração de recursos por lista; as métricas desta
  seção servem para diagnosticar **por que** ele se move entre 2018 e 2022.